# River Flow Forecasting

## Project Overview

Forecasts **daily river flow** (cumecs / m³/s) over a 14-day horizon. Synthetic data spans 2 years with seasonal snowmelt/rainfall patterns, flood events, and baseflow recession.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily river flow data, predict the next 14 days. Water resource managers need flow forecasts for flood warnings, reservoir operations, irrigation allocation, and environmental flows.

## Why This Project Matters

River flow forecasting saves lives and infrastructure. Flood early warnings need 3-14 day lead times. Reservoir operators balance flood control, water supply, and hydropower generation using inflow forecasts. Droughts require early detection for water rationing.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily river flow
- Strong seasonal pattern (spring snowmelt peak, winter baseflow)
- Rainfall-driven flood events
- Baseflow recession during dry periods
- No weekly pattern

| Column | Description |
|--------|-------------|
| `date` | Date |
| `flow_cumecs` | Daily average river flow (m³/s) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'flow_cumecs'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Seasonal: spring snowmelt peak
seasonal = 50 + 40 * np.sin(2 * np.pi * (t - 60) / 365)

# Baseflow with recession
baseflow = 20 + 0.005 * t

# Rainfall-driven flood events
floods = np.zeros(N_POINTS)
for s in range(0, N_POINTS, 60):
    flood_day = min(s + np.random.randint(10, 50), N_POINTS - 5)
    peak = np.random.uniform(30, 80)
    for j in range(min(5, N_POINTS - flood_day)):
        floods[flood_day + j] = peak * np.exp(-0.5 * j)  # exponential recession

noise = np.random.normal(0, 5, N_POINTS)
flow_cumecs = seasonal + baseflow + floods + noise
flow_cumecs = np.maximum(flow_cumecs, 5).round(1)

df = pd.DataFrame({'date': dates, 'flow_cumecs': flow_cumecs})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,flow_cumecs
0,2022-01-01,34.4
1,2022-01-02,35.2
2,2022-01-03,29.0
3,2022-01-04,44.2
4,2022-01-05,37.0


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('flow_cumecs Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("flow_cumecs Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

flow_cumecs Statistics:
count    730.000000
mean      74.181918
std       29.923003
min       18.300000
25%       44.950000
50%       74.300000
75%      101.975000
max      166.300000
Name: flow_cumecs, dtype: float64

CV: 0.403
Skewness: 0.084


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       21.6 | RMSE:       31.8 | MAPE: 33.67%
Seasonal Naive (7)             | MAE:       17.0 | RMSE:       27.8 | MAPE: 24.52%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared       RMSE  Time Taken
Model                                                                          
KernelRidge                      8331.445604 -639.803508  80.486026    0.050898
QuantileRegressor                2534.428243 -193.879096  44.385455    0.058713
DummyRegressor                   2176.643310 -166.357178  41.132037    0.007771
GaussianProcessRegressor          733.096457  -55.315112  23.860004    0.047070
MLPRegressor                      127.532888   -8.733299   9.919461    0.306553
NuSVR                             119.127146   -8.086704   9.584318    0.019696
SVR                                96.822569   -6.370967   8.632176    0.024775
Lars                               75.404934   -4.723456   7.606541    0.017501
ExtraTreeRegressor                 60.256092   -3.558161   6.788172    0.008741
DecisionTreeRegressor              55.747885   -3.211376   6.524843    0.014554


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        3.7 | RMSE:        4.3 | MAPE: 10.68%
FLAML Test (catboost)          | MAE:       10.0 | RMSE:       20.5 | MAPE: 14.15%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       15.8 | RMSE:       27.3 | MAPE: 21.81%
SF AutoETS                     | MAE:       15.8 | RMSE:       27.3 | MAPE: 21.71%
SF SeasonalNaive               | MAE:       16.8 | RMSE:       28.9 | MAPE: 23.26%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  3.672116  4.291666 10.681071
FLAML Test (catboost) 10.003597 20.526601 14.153320
           SF AutoETS 15.805383 27.284951 21.714129
         SF AutoARIMA 15.819398 27.280516 21.812224
     SF SeasonalNaive 16.828571 28.909909 23.258444
   Seasonal Naive (7) 17.000000 27.781314 24.520419
   Naive (Last Value) 21.628571 31.829030 33.666017

Top 3:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  3.672116  4.291666 10.681071
FLAML Test (catboost) 10.003597 20.526601 14.153320
           SF AutoETS 15.805383 27.284951 21.714129


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 6.34, Std: 19.52


## Interpretation and Business Insight

- River flow peaks in spring due to snowmelt
- Flood events create sharp peaks with exponential recession
- Baseflow provides the minimum flow during dry periods
- No weekly pattern — purely driven by hydrology and weather
- Recession curves after flood events are highly predictable

## Limitations

1. Synthetic — real flow depends on catchment geology, land use, upstream dams
2. No rainfall input data (key driver)
3. No snowpack/snowmelt modeling
4. Single station — upstream/downstream relationships not modeled
5. No reservoir operations effects

## How to Improve This Project

1. Add rainfall and temperature as input features
2. Include snowpack data for snowmelt forecasting
3. Model upstream-downstream flow routing
4. Add soil moisture for rainfall-runoff modeling
5. Use physics-based hydrological models alongside ML

## Production Considerations

- Daily flow forecasting for flood early warning
- Integration with reservoir operation optimization
- Drought monitoring and water rationing triggers
- Environmental flow compliance monitoring

## Common Mistakes

1. Ignoring rainfall — it's the primary flow driver
2. Not modeling recession curves for flood events
3. Using simple time-series models when rainfall-runoff physics matters
4. Treating all flow as the same (snowmelt vs rainfall vs baseflow)

## Mini Challenge / Exercises

1. Add synthetic rainfall and correlate with flow peaks
2. Fit an exponential recession model after flood events
3. Build a flood threshold exceedance probability forecast
4. Compare spring (snowmelt) vs autumn (rainfall) flood characteristics

## Final Summary / Key Takeaways

1. River flow forecasting is critical for flood warning and water management
2. Seasonal snowmelt provides the dominant annual pattern
3. Flood events are driven by rainfall (not included)
4. Recession curves are highly predictable and useful for forecasting
5. Real deployment needs rainfall inputs and hydrological modeling

In [18]:
import json
metrics = {
    'project': 'River Flow Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== River Flow Forecasting — Complete ===")

Metrics saved.

=== River Flow Forecasting — Complete ===
